# 🚘 PlateTrack AI - Cloud Backend (Google Colab)
This notebook runs the **FastAPI + YOLO11 + EasyOCR** backend with **12.7 GB Free RAM** and generates a live public HTTPS URL via Cloudflare Tunnel for your Vercel frontend.

In [ ]:
# Step 1: Clone repository and install dependencies
!git clone https://github.com/mayankagarwal222005-art/himaliyan-enterprice.git
%cd himaliyan-enterprice

!pip install -q fastapi uvicorn[standard] python-multipart ultralytics easyocr openpyxl pandas opencv-python-headless

# Install Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# Step 2: Start server and get live HTTPS link for Vercel
import subprocess
import threading
import time
import re

def start_uvicorn():
    subprocess.run(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"])

threading.Thread(target=start_uvicorn, daemon=True).start()
print("⏳ Starting FastAPI Server...")
time.sleep(5)

print("🚀 Starting Cloudflare Tunnel...")
process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

url_found = False
for line in process.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match and not url_found:
        public_url = match.group(0)
        url_found = True
        print("\n" + "="*70)
        print("🎉 YOUR LIVE BACKEND URL IS:")
        print(public_url)
        print("="*70)
        print(f"👉 Add this URL to Vercel Environment Variables:")
        print(f"   BACKEND_URL = {public_url}")
        print("="*70 + "\n")